# AM 11B Lab Activity 5: Present Value of a Revenue Stream

**Week 8 — built on: Substitution, Integration by Parts, and Other Integration Techniques**

**Authors:** Julie Simons, with Claude (Sonnet 5, Anthropic), via Claude Code — September 2026

*With special thanks to Mariana Bessa and prior AM 11B students, whose earlier notebooks inspired this material, and to Eric Van Dusen and the [Data 88E](https://github.com/data-88e) course for inspiration.*

![A balance scale weighing a clock against a pile of coins, illustrating the time value of money](https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b9/Time_Money_Balance.jpg/960px-Time_Money_Balance.jpg)

*Photo: Artistosteles, CC BY-SA 4.0, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Time_Money_Balance.jpg)*

<!-- canvas-group-instructions -->
## Group Submission Instructions

**Joining your lab group on Canvas**

1. Agree with your partners on a group number (e.g., "Lab Group 7").
2. In our Canvas course, click **People**, then the **Groups** tab.
3. Find your group and click **Join**. **Every member must do this themselves.** You can't add your partners.
4. To check, click the group name and make sure all your partners are listed. If you joined the wrong group, click **Leave** and join the right one.
5. When you submit, **only one person uploads the notebook**. It counts for everyone in the group.

Please join your group before submitting your first lab. If you submit before joining, only you will get credit.

Please write the name(s) of your group (or yourself) below by double-clicking on this text.

**NAME(S):** *Replace this text with your name(s)*

## Learning Goals

By the end of this lab you should be able to:
1. Set up an integral that combines an exponential discount factor with a polynomial revenue function, and find its antiderivative by hand using integration by parts.
2. Evaluate a definite integral symbolically and confirm it numerically.
3. Interpret the result as the **present value** of a stream of future revenue, and contrast it with plain (undiscounted) total revenue over the same period.
4. See directly that present value does *not* scale linearly with the time horizon — doubling the horizon does not double the present value.
5. Explore, interactively, how present value responds to how far into the future you look — and notice that it grows by *less and less* per extra decade.

### Why economists care
A dollar earned five years from now is worth less than a dollar today (money can be invested and grow in the meantime). "Discounting" future cash flows back to today's dollars is central to valuing companies, investment projects, and government programs — and it's a direct application of the integration techniques you just learned.

## Setup

Run the cell below to load the tools we'll use today.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy import integrate
from ipywidgets import interact, IntSlider
import random

sp.init_printing()
plt.rcParams['figure.figsize'] = (9, 5)
%matplotlib inline

def checkpoint(is_correct, hint=""):
    if is_correct:
        emoji = random.choice(["✅", "🚀", "⭐"])
        msg = random.choice([
            "Great job! You nailed it.",
            "You did it!",
            "Nailed it — nice work.",
            "Yes! That's exactly right.",
            "Correct! You're crushing this.",
        ])
        print(f"{emoji} {msg}")
    else:
        print("❌ Try again!" + (f" {hint}" if hint else ""))

print("Setup complete!")

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

## The Scenario

A small business expects its annual revenue to grow steadily over the next several years, following

$$R(t) = 1000 + 50t \quad (\text{dollars per year}, \ t \text{ years from now})$$

Because money now is worth more than money later, we discount future revenue by a continuous discount rate $r = 0.05$ (5% per year), using the discount factor $e^{-rt}$. The **present value** of all revenue earned between now and year $T$ is:

$$PV = \int_0^T R(t)\, e^{-rt}\, dt = \int_0^T (1000 + 50t)\, e^{-0.05t}\, dt$$

Notice the integrand is a **product of a polynomial and an exponential** — exactly the setup where integration by parts shines (you're integrating $u \, dv$ with $u = 1000+50t$ growing simple under differentiation, and $dv = e^{-rt}dt$ easy to integrate).

### Step 1: Find the antiderivative — try it by hand first

**Before running any code**, find this antiderivative by hand using integration by parts, $\int u\,dv = uv - \int v\,du$. A good split:

- $u = 1000 + 50t$ (differentiates to something simpler: $du = 50\,dt$)
- $dv = e^{-0.05t}\,dt$ (integrates easily: $v = -20e^{-0.05t}$, since dividing by $-0.05$ is the same as multiplying by $-20$)

Work through the formula and simplify. Write your final antiderivative below.

</div>

*Replace this text with your hand-derived antiderivative.*

Now run the cell below to have `sympy` find the antiderivative too, and check it against what you found on paper.

In [ ]:
t, r = sp.symbols('t r', positive=True)
R = 1000 + 50*t
rate = sp.Rational(5, 100)
integrand = R * sp.exp(-rate*t)

antiderivative = sp.integrate(integrand, t)
print("Antiderivative:", sp.simplify(antiderivative))

expected_antiderivative = 1000*(-t - 40)*sp.exp(-t/20)
checkpoint(
    sp.simplify(antiderivative - expected_antiderivative) == 0,
    hint="Double check your u, dv split and the sign when you substitute v back in."
)

### Step 2: Evaluate present value over the first 5 years

Quick reminder, since we'll do this again a few times in this lab: **symbolic** computation (`sympy`) works from the actual algebraic formula and gives an exact answer, the way you'd solve it by hand. **Numerical** computation (`scipy.integrate.quad`) never touches the formula — it estimates the area under the curve by crunching a huge number of thin slices, entirely independently of the algebra. Run the cell below to compute the present value both ways and confirm they land on the same number.

In [ ]:
PV_5 = sp.integrate(integrand, (t, 0, 5))
print(f"PV over 5 years (symbolic, sympy):   {sp.simplify(PV_5)} = ${float(PV_5):.2f}")

f = lambda tt: (1000 + 50*tt) * np.exp(-0.05*tt)
val, err = integrate.quad(f, 0, 5)
print(f"PV over 5 years (numerical, scipy):  ${val:.2f}")

checkpoint(abs(float(PV_5) - 4953.96) < 0.01 and abs(val - 4953.96) < 0.01, hint="PV over 5 years should be about $4953.96.")

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

### Question 5.2 (Computational Check): Undiscounted Total Revenue

Before we look at how discounting changes the picture, let's compute a simpler baseline: the plain (undiscounted) total revenue earned over the first 5 years, with no discount factor at all:

$$\text{Total Revenue} = \int_0^5 R(t)\, dt = \int_0^5 (1000 + 50t)\, dt$$

**Before running any code**, evaluate this by hand using just the power rule — there's no exponential here, so no integration by parts needed.

Which of the following is closest to your answer?

- A) \$4,953.96
- B) \$5,000.00
- C) \$5,625.00
- D) \$6,250.00

</div>

Run the cell below to check your hand computation.

In [ ]:
total_revenue_5 = sp.integrate(R, (t, 0, 5))
print(f"Undiscounted total revenue over 5 years: {total_revenue_5} = ${float(total_revenue_5):.2f}")
print(f"Compare to the discounted PV over 5 years from Step 2:  ${float(PV_5):.2f}")

Now pick the letter that matches the value you just saw.

In [ ]:
mc_answer_5_2 = None   # EDIT HERE -- set to "A", "B", "C", or "D"
assert mc_answer_5_2 in ["A", "B", "C", "D"], 'Set mc_answer_5_2 to one of "A", "B", "C", "D".'

checkpoint(
    mc_answer_5_2 == "C",
    hint="This is just the integral of (1000+50t) from 0 to 5, with no discount factor -- recheck the power rule integration."
)

## Visualizing Discounting

Run the cell below to see two curves: the *undiscounted* revenue $R(t)$ (what the business earns each year), and the *discounted* revenue $R(t)e^{-rt}$ (what that year's revenue is worth in today's dollars). The shaded area under the discounted curve, from $0$ to $5$, is the present value you just computed.

In [ ]:
t_vals = np.linspace(0, 15, 200)
R_vals = 1000 + 50*t_vals
R_disc_vals = R_vals * np.exp(-0.05*t_vals)

plt.figure()
plt.plot(t_vals, R_vals, 'g--', linewidth=2, label='Revenue $R(t)$ (undiscounted)')
plt.plot(t_vals, R_disc_vals, 'b-', linewidth=2, label='Discounted revenue $R(t)e^{-rt}$')
t_fill = np.linspace(0, 5, 100)
plt.fill_between(t_fill, (1000+50*t_fill)*np.exp(-0.05*t_fill), alpha=0.3, color='blue', label='PV over 5 years')
plt.xlabel('Year (t)')
plt.ylabel('Dollars')
plt.title('Revenue vs. Discounted Revenue')
plt.legend()
plt.show()

### Question 5.1: Why the Gap Grows

The undiscounted and discounted curves start at the same point ($t=0$) but drift apart as $t$ increases. Explain, in terms of the discount factor $e^{-rt}$, why the gap between the two curves grows wider the farther out in time you look.

*Replace this text with your answer.*

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

### Question 5.3 (Computational Check): Present Value Over 10 Years

Let's use the *same* antiderivative from Step 1 to find present value over a longer horizon, before you explore even more horizons interactively with the slider below.

**Before running any code**, evaluate $F(10) - F(0)$ using the antiderivative you already found. (Or just reason about it: will doubling the time horizon double the present value? Discounting isn't linear in time, so think about whether that assumption holds up.)

Which of the following is closest to the present value of revenue over the first 10 years?

- A) \$4,953.96
- B) \$9,673.47
- C) \$9,907.93
- D) \$12,500.00

</div>

Run the cell below to check your hand computation.

In [ ]:
PV_10 = antiderivative.subs(t, 10) - antiderivative.subs(t, 0)
print(f"Present value over 10 years: {sp.simplify(PV_10)} = ${float(PV_10):.2f}")

Now pick the letter that matches the value you just saw.

In [ ]:
mc_answer_5_3 = None   # EDIT HERE -- set to "A", "B", "C", or "D"
assert mc_answer_5_3 in ["A", "B", "C", "D"], 'Set mc_answer_5_3 to one of "A", "B", "C", "D".'

checkpoint(
    mc_answer_5_3 == "B",
    hint="Evaluate the Step 1 antiderivative at t=10 and at t=0, then subtract -- don't just double the 5-year answer."
)

<div style="background-color:#e7f1fc; border-left:6px solid #4a90d9; border-radius:4px; padding:4px 18px 4px 18px; color:#0b3d78; margin:12px 0;">

## Your Turn: Explore the Time Horizon with a Slider

Rather than editing a number and re-running by hand, let's make the horizon $T$ interactive. The cell below reuses the exact antiderivative you found in Step 1 (so it's the Fundamental Theorem of Calculus doing the work — evaluate the antiderivative at $T$ and at $0$, then subtract) and lets you **drag a slider** to see present value update live as $T$ changes, from 1 year out to 40 years out.

Run the cell, then drag the slider around.

</div>

In [ ]:
F_numeric = sp.lambdify(t, antiderivative, 'numpy')   # turn the Step 1 antiderivative into a fast numeric function

def pv_demo(T=5):
    PV_T = float(F_numeric(T) - F_numeric(0))       # FTC: the symbolic antiderivative, evaluated at T minus at 0
    PV_check, _ = integrate.quad(f, 0, T)            # a fully independent numerical (slice-and-sum) check

    t_smooth = np.linspace(0, 40, 300)
    disc_smooth = (1000 + 50*t_smooth) * np.exp(-0.05*t_smooth)

    plt.figure(figsize=(9, 5))
    plt.plot(t_smooth, disc_smooth, 'b-', linewidth=2, label='Discounted revenue $R(t)e^{-rt}$')
    t_fill = np.linspace(0, T, 100)
    plt.fill_between(t_fill, (1000 + 50*t_fill)*np.exp(-0.05*t_fill), alpha=0.3, color='blue',
                      label=f'PV over {T} years')
    plt.xlabel('Year (t)')
    plt.ylabel('Dollars')
    plt.title(f"Present Value Over {T} Years")
    plt.legend()
    plt.ylim(0, 1100)
    plt.xlim(0, 40)
    plt.show()

    print(f"T = {T:>2} years  ->  Present Value (from the antiderivative) = ${PV_T:,.2f}   |   independent numerical check (scipy) = ${PV_check:,.2f}")
    checkpoint(abs(PV_T - PV_check) < 0.01)

interact(pv_demo, T=IntSlider(min=1, max=40, step=1, value=5, description='Horizon T (yrs)'));

*(If the slider doesn't appear, just edit `value=5` in the cell above to a few different numbers by hand and re-run — the plot and printed present value will update either way.)*

### Question 5.2: Diminishing Additions

Drag the slider to read off the present value at $T=10$, $T=20$, $T=30$, and $T=40$. Compute how much value is **added in each 10-year window**: from $T=0$ to $10$, from $10$ to $20$, from $20$ to $30$, and from $30$ to $40$. What pattern do you notice in these four amounts, and how does it connect to the shape of the discounted-revenue curve shown by the slider?

*Replace this text with your answer.*

## Reflection

**In 1-2 sentences:** If the discount rate $r$ were higher (say 10% instead of 5%), would you expect present value over a fixed horizon to go up or down? Briefly explain why, using the formula $e^{-rt}$.

*Replace this text with your answer.*

---

## Submission

1. Make sure you've run all cells in order (**Runtime/Kernel → Restart and run all** is a good final check).
2. Answer the short reflection question(s) above by editing the markdown cells.
3. Go to **File → Download** to save the `.ipynb` file to your computer.
4. Upload the file to Canvas under this Lab Activity.

Only one person per group needs to submit if you worked together — just make sure everyone's name is listed at the top.

**This lab is designed to take about 30 minutes.** If you're stuck for more than a few minutes on any one cell, that's your cue to ask a neighbor, a TA, or post on Ed Discussion — don't burn your whole discussion period on one line of code.
